In [23]:
# 1
import time
from typing import Tuple, List, Dict
import requests
import pandas as pd

URL = "https://api.musinsa.com/api2/dp/v2/plp/goods"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}
DELAY = 0.7
MAX_PAGE = 10

def fetch(page: int, next_url: str = None) -> dict:
    if next_url:
        response = requests.get(next_url, headers=HEADERS)
    else:
        params = {
            'gf': 'A', 
            'sortCode': 'POPULAR', 
            'category': '001',
            'size': '60', 
            'caller': 'CATEGORY', 
            'seen': '121',
            'page': page
        }
        response = requests.get(URL, params=params, headers=HEADERS)
        
    return response.json()

def parse(data: dict) -> Tuple[List[Dict], str]:
    if not data or not data.get('data'):
        return [], None

    rows = []
    goods_list = data['data'].get('list', [])
    
    for item in goods_list:
        rows.append({
            "브랜드명": item.get('brandName', ''),
            "제품명": item.get('goodsName', ''),
            "원래가격": item.get('normalPrice', ''), 
            "할인가격": item.get('finalPrice', ''),       
            "리뷰수": item.get('reviewCount', 0),
            "리뷰점수": item.get('reviewScore', 0)
        })
        
    next_url = data['data']['pagination'].get('nextPageUrl')
    
    return rows, next_url

results = []
current_next_url = None

for page in range(1, MAX_PAGE + 1):
    data = fetch(page, current_next_url)
    rows, current_next_url = parse(data)
    
    if not rows:
        print(f"{page}페이지가 비어 있습니다")
        break
        
    results.extend(rows)
    print(f"{page}페이지 수집 완료. 누적 {len(results)}건")
    
    if not current_next_url:
        print("마지막 페이지에 도달했습니다")
        break
        
    time.sleep(DELAY)

df = pd.DataFrame(results)
df.to_csv("musinsa.csv", index=False, encoding="utf-8-sig")
print("musinsa.csv 저장 완료")

1페이지 수집 완료. 누적 60건
2페이지 수집 완료. 누적 120건
3페이지 수집 완료. 누적 180건
4페이지 수집 완료. 누적 240건
5페이지 수집 완료. 누적 300건
6페이지 수집 완료. 누적 360건
7페이지 수집 완료. 누적 420건
8페이지 수집 완료. 누적 480건
9페이지 수집 완료. 누적 540건
10페이지 수집 완료. 누적 600건
musinsa.csv 저장 완료


In [22]:
# 2
import time
from typing import Tuple, List, Dict
import requests
import pandas as pd

URL = "https://www.rocketpunch.com/api/proxy/jobs"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "x-rocket-app-key": "077cd9d8-7237-4bee-b2d3-77690d163cca",
    "x-rocket-client-type": "WEB",
    "x-rocket-device-type": "PC",
    "x-rocket-os-type": "WIN",
}

DELAY = 1.0
MAX_PAGE = 10

def fetch(page_token: str = None) -> dict:
    params = {}
    
    if page_token:
        params['pageToken'] = page_token
        
    response = requests.get(URL, headers=HEADERS, params=params, timeout=10)
    response.raise_for_status() 
    
    return response.json()

def parse(data: dict) -> Tuple[List[Dict], str]:
    rows = []
    items = data.get('items', [])
    
    for item in items:
        desc = item.get('description')
        if not desc:
            desc = ""
            
        rows.append({
            "기업명": item.get('companyName', ''),
            "공고명": item.get('title', ''),
            "요약": desc,
            "업무형태": item.get('workType', '')
        })
        
    next_token = data.get('pageToken')
    
    return rows, next_token

def main():
    results = []
    current_token = None
    
    for page in range(1, MAX_PAGE + 1):
        data = fetch(current_token)
        rows, current_token = parse(data)
        
        if not rows:
            print(f"{page}페이지 파싱 결과가 없습니다")
            break
            
        results.extend(rows)
        print(f"{page}페이지 수집 완료 · 누적 {len(results)}건")
        
        if not current_token:
            print("마지막 페이지에 도달했습니다")
            break
            
        time.sleep(DELAY)

    df = pd.DataFrame(results)
    df.to_csv("rocketpunch.csv", index=False, encoding="utf-8-sig")
    print("rocketpunch.csv 저장 완료")

if __name__ == "__main__":
    main()

1페이지 수집 완료 · 누적 20건
2페이지 수집 완료 · 누적 40건
3페이지 수집 완료 · 누적 60건
4페이지 수집 완료 · 누적 80건
5페이지 수집 완료 · 누적 100건
6페이지 수집 완료 · 누적 120건
7페이지 수집 완료 · 누적 140건
8페이지 수집 완료 · 누적 160건
9페이지 수집 완료 · 누적 180건
10페이지 수집 완료 · 누적 200건
rocketpunch.csv 저장 완료
